In [7]:
#!pip install pdfplumber -q

In [ ]:
import pdfplumber, re, json, statistics
from pathlib import Path

# 실제 PDF는 프로젝트 .venv 폴더에 있어 절대경로로 지정
PDF_PATH = "/Users/gimseoyeon/Downloads/AICPP/.venv/고영향_인공지능_판단_가이드라인.pdf"
# 결과물은 레포 안 data/processed 에 바로 저장 → git add 만 하면 됨
OUTPUT_PATH = "/Users/gimseoyeon/Downloads/AICPP/data/processed/판단가이드라인_chunks.jsonl"
CHUNK_SIZE = 600
OVERLAP = 100

assert Path(PDF_PATH).exists(), f"PDF 없음: {PDF_PATH}"
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
print("PDF 확인 완료:", PDF_PATH)


PDF 확인 완료: /Users/gimseoyeon/Downloads/AICPP/.venv/고영향_인공지능_판단_가이드라인.pdf


## 추출 + 클리닝

In [3]:
def extract_and_clean(pdf_path):
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t and t.strip():
                texts.append(t.strip())
    full = "\n\n".join(texts)
    # 클리닝
    full = re.sub(r'고영향 인공지능 판단 가이드라인\s*', '', full)
    full = re.sub(r'^\s*[-–]?\s*\d+\s*[-–]?\s*$', '', full, flags=re.MULTILINE)
    full = full.replace('Ÿ', '•').replace('ㅅ', '').replace('►', '▶')
    full = re.sub(r'[ \t]{2,}', ' ', full)
    full = re.sub(r'\n{3,}', '\n\n', full)
    print(f"유효 페이지: {len(texts)}, 총 {len(full)}자")
    return full

full = extract_and_clean(PDF_PATH)
print(full[:400])


유효 페이지: 171, 총 189316자
1.1.1 고영향 인공지능 가이드라인의 수립 배경
▶ 인공지능기본법 제1조(목적)
이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함
으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데
이바지함을 목적으로 한다.
「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」(2026.1.22.시행)(이하 ‘인공지능기본법’)은
인공지능의 건전한 발전을 지원하고 인공지능사회의 신뢰 기반 조성에 필요한 기본적인 사항을 규정
하고자 제정되었다.
• 인공지능기본법에서는 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을
초래할 우려가 있는 분야의 고영향 인공지능사업자에게 일정한 책무를 부여하고 인공지능
이용자를 보호하고자 한다.
▶ 인공지능기


## 섹션 분리

In [4]:
SECTION_PAT = re.compile(r'(?m)^(\d+(?:[.-]\d+){1,3})[.\s]\s*(.{4,80})$')

def split_sections(text):
    matches = list(SECTION_PAT.finditer(text))
    sections = []
    for i, m in enumerate(matches):
        sid = m.group(1).strip()
        title = m.group(2).strip()
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        body = text[start:end].strip()
        if len(body) < 30:
            continue
        sections.append({
            "section_id": sid,
            "title": title,
            "body": body,
            "has_legal_ref": bool(re.search(r'▶|제\d+조', body)),
            "char_count": len(body)
        })
    print(f"섹션: {len(sections)}개")
    lengths = [s["char_count"] for s in sections]
    print(f"평균 {statistics.mean(lengths):.0f}자, 600자 초과: {sum(1 for l in lengths if l>600)}개")
    return sections

sections = split_sections(full)
for s in sections[:6]:
    print(f"  [{s['section_id']}] {s['title'][:40]} ({s['char_count']}자)")


섹션: 73개
평균 2569자, 600자 초과: 63개
  [1.1.1] 고영향 인공지능 가이드라인의 수립 배경 (810자)
  [1-1-1] (참고) 인공지능기본법의 특징 (1197자)
  [1-1-2] (참고) 고영향 인공지능의 확인 절차 (2142자)
  [1.2.1] 인공지능기본법 제2조(정의)에 따른 주요 용어 (1313자)
  [1.2.2] 인공지능시스템의 개념 (1040자)
  [1.2.3] 인공지능기본법의 인적 적용 범위 (4012자)


## 오버랩 청킹

In [5]:
def overlap_chunk(text, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            cut = text.rfind('\n', start, end)
            if cut == -1:
                cut = text.rfind('. ', start, end)
            # 줄바꿈이 뒤쪽 절반에 있을 때만 적용 (end 가 start 근처로 당겨지는 것 방지)
            if cut != -1 and cut > start + size // 2:
                end = cut + 1
        chunks.append(text[start:end].strip())
        if end >= n:
            break
        start = max(end - overlap, start + 1)   # 항상 전진 보장 → 무한루프 방지
    return [c for c in chunks if len(c) > 20]

def build_chunks(sections):
    all_chunks = []
    for sec in sections:
        if sec["char_count"] <= CHUNK_SIZE:
            all_chunks.append({
                "chunk_id": f"{sec['section_id']}_0",
                "section_id": sec["section_id"],
                "title": sec["title"],
                "content": f"[{sec['section_id']}] {sec['title']}\n{sec['body']}",
                "has_legal_ref": sec["has_legal_ref"],
                "출처": "고영향 인공지능 판단 가이드라인",
                "청크유형": "섹션"
            })
        else:
            for j, chunk_text in enumerate(overlap_chunk(sec["body"])):
                all_chunks.append({
                    "chunk_id": f"{sec['section_id']}_{j}",
                    "section_id": sec["section_id"],
                    "title": sec["title"],
                    "content": f"[{sec['section_id']}] {sec['title']} (파트{j+1})\n{chunk_text}",
                    "has_legal_ref": sec["has_legal_ref"] and j == 0,
                    "출처": "고영향 인공지능 판단 가이드라인",
                    "청크유형": "섹션(분할)"
                })
    print(f"최종 청크: {len(all_chunks)}개")
    print(f"  단일: {sum(1 for c in all_chunks if c['청크유형']=='섹션')}개")
    print(f"  분할: {sum(1 for c in all_chunks if '분할' in c['청크유형'])}개")
    print(f"  법조항 포함: {sum(1 for c in all_chunks if c['has_legal_ref'])}개")
    return all_chunks

chunks = build_chunks(sections)


최종 청크: 410개
  단일: 10개
  분할: 400개
  법조항 포함: 50개


## 저장

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"저장 완료 → {OUTPUT_PATH}")
lengths = [len(c["content"]) for c in chunks]
print(f"평균 청크 길이: {sum(lengths)//len(lengths)}자")
print(f"최소: {min(lengths)}자 / 최대: {max(lengths)}자")


저장 완료 → /Users/gimseoyeon/Downloads/AICPP/data/processed/판단가이드라인_chunks.jsonl
평균 청크 길이: 572자
최소: 152자 / 최대: 661자
